# Five fourth-order models over 200 cycles: calculation

This notebook is the compute-only half of the five-method comparison. It defines the complete reproducible experiment, audits the SDIRK4 S54b tableau, performs the two adaptive references and all five fixed-step integrations, and writes every trajectory and diagnostic to `results.csv`.

It intentionally contains no plotting or animation code, so it is the only notebook that needs to run on the AWS compute environment.

ABBA2 and ABBA4 now share preparation, the integration loop, state handling, nonlinear convergence control, and accepted-step records. This study keeps ABBA4's single projection around the complete composition. The public simulation API, result labels, and diagnostic keys remain compatible; projection diagnostics describe accepted integration steps, independently of the saved-time grid.

## Reproducible long-time experiment

The dominant measured mode is normalized to one cycle per unit time. The interval $[0,200]$ therefore contains 200 cycles. Both the effective integration step and saved-state interval are $0.02$: every method takes exactly 50 complete steps per cycle, 10,000 steps in total, and returns 10,001 aligned states.

DOP853 supplies the accuracy reference and a tighter Radau solve independently audits its numerical floor. At long times, pointwise trajectory errors may approach the periodic-domain scale, so they should be interpreted together with the physical-Hamiltonian error history and not as a conserved-energy test: the measured potential is explicitly time dependent.

The output contract is one schema-versioned, wide CSV row per saved time. Its explicit columns include DOP853 and Radau states, all five numerical trajectories, trajectory and Hamiltonian errors, and step-aligned Newton and projection diagnostics. Configuration, timings, summaries, and the execution log are stored as JSON metadata in the first row.

In [1]:
from contextlib import redirect_stderr
from pathlib import Path
import sys

from IPython.display import Markdown, display
import numpy as np

from diagnostics import write_five_method_comparison_csv
from diagnostics.paths import find_project_root
from potential import GC2DH5Metadata, load_gc2d_h5_potential
from simulation import SDIRK4_TABLEAU_A, SDIRK4_TABLEAU_B, SDIRK4_TABLEAU_C
from studies import (
    FIVE_METHOD_COMPARISON_METHODS,
    FIVE_METHOD_IMPLICIT_METHODS,
    FiveMethodComparisonConfig,
    domain_center,
    latin_hypercube_gc_configuration_with_near_center,
    run_five_method_comparison,
)

In [2]:
# Measured, nondimensionalized GC2D potential used by the project.
project_root = find_project_root(Path.cwd())
notebook_directory = (
    project_root
    / "notebooks/developements/energy/compare_five_order4_models_with_sdirk_rk4"
)
notebook_directory.mkdir(parents=True, exist_ok=True)
results_path = notebook_directory / "results.csv"
execution_log_path = notebook_directory / "calculation.log"
data_path = project_root / "data/potential/V1/PHI_2.h5"
magnetic_field = 1.5
characteristic_length = 0.06
mode_selection = (0, 1)
interpolation_order = 3
if not data_path.is_file():
    raise FileNotFoundError(f"Measured HDF5 potential not found: {data_path}")

# Physical parameters and three spatially distributed initial trajectories.
rho = 0.3
coupling_frequency = float(np.pi / 8.0)
trajectory_count = 3
initial_condition_seed = 20260905
domain_margin_fraction = 0.05
near_center_offset_fraction = (0.08, -0.06)

# Two hundred cycles with fifty effective and saved steps per cycle.
cycle_duration = 1.0
steps_per_cycle = 50
t_span = (0.0, 200.0)
integration_step = cycle_duration / steps_per_cycle
save_interval = integration_step

# Common Newton controls.
newton_absolute_tolerance = 1e-13
newton_relative_tolerance = 1e-12
newton_max_iterations = 40
jacobian_relative_step = float(np.cbrt(np.finfo(float).eps))

# DOP853 reference and tighter independent Radau audit.
reference_relative_tolerance = 1e-10
reference_absolute_tolerance = 1e-12
reference_maximum_step = 0.025
audit_relative_tolerance = 1e-11
audit_absolute_tolerance = 1e-13
audit_maximum_step = 0.0125
timing_warmups = 0
timing_repeats = 3

potential = load_gc2d_h5_potential(
    data_path,
    B=magnetic_field,
    characteristic_length=characteristic_length,
    indx=mode_selection,
    interpolation_order=interpolation_order,
)
potential_metadata = potential.metadata
assert isinstance(potential_metadata, GC2DH5Metadata)
initial_configuration = latin_hypercube_gc_configuration_with_near_center(
    potential,
    particle_count=trajectory_count,
    seed=initial_condition_seed,
    center_offset_fraction=near_center_offset_fraction,
    domain_margin_fraction=domain_margin_fraction,
)
config = FiveMethodComparisonConfig(
    rho=rho,
    coupling_frequency=coupling_frequency,
    t_span=t_span,
    integration_step=integration_step,
    save_interval=save_interval,
    absolute_tolerance=newton_absolute_tolerance,
    relative_tolerance=newton_relative_tolerance,
    max_iterations=newton_max_iterations,
    jacobian_relative_step=jacobian_relative_step,
    reference_relative_tolerance=reference_relative_tolerance,
    reference_absolute_tolerance=reference_absolute_tolerance,
    reference_maximum_step=reference_maximum_step,
    audit_relative_tolerance=audit_relative_tolerance,
    audit_absolute_tolerance=audit_absolute_tolerance,
    audit_maximum_step=audit_maximum_step,
    timing_warmups=timing_warmups,
    timing_repeats=timing_repeats,
    distance_convention="periodic",
    progress=True,
)

assert config.step_count == 10_000
assert config.output_sample_count == 10_001
assert initial_configuration.initial_state is not None
assert initial_configuration.initial_state.size == 2 * trajectory_count
near_center_coordinates = (
    np.asarray(domain_center(potential), dtype=float)
    + np.asarray(near_center_offset_fraction) * potential.grid.period
)
np.testing.assert_allclose(
    (initial_configuration.initial_state[0], initial_configuration.initial_state[trajectory_count]),
    near_center_coordinates,
)

display(Markdown(
    f"**Resolved grid:** `{config.step_count}` effective steps of `{integration_step:g}`, "
    f"`{config.output_sample_count}` saved states, `{steps_per_cycle}` steps per cycle, "
    f"and `{trajectory_count}` trajectories on `[{t_span[0]:g}, {t_span[1]:g}]`.\n\n"
    f"**Measured potential:** `{potential_metadata.source_path}`, grid `{potential.grid.shape[0]} x {potential.grid.shape[1]}`, "
    f"selected source fields `{potential_metadata.source_field_indices.tolist()}` and normalized frequencies `{potential.frequencies.tolist()}`."
))

**Resolved grid:** `10000` effective steps of `0.02`, `10001` saved states, `50` steps per cycle, and `3` trajectories on `[0, 200]`.

**Measured potential:** `/home/juan/Proyectos/GC2D_intranet/data/potential/V1/PHI_2.h5`, grid `256 x 256`, selected source fields `[15]` and normalized frequencies `[1.0]`.

## SDIRK4 S54b: order-four and non-geometric certificates

The five-stage tableau has common diagonal $1/4$ and its last row equals the weights. The next cell verifies all eight classical order conditions through order four. It also evaluates the Runge--Kutta symplecticity conditions

$$b_i a_{ij}+b_j a_{ji}-b_i b_j=0$$

and the adjoint-tableau symmetry conditions. Both structural defects are substantially nonzero; they do not depend on the integration step or Newton tolerance.

In [3]:
A = SDIRK4_TABLEAU_A
b = SDIRK4_TABLEAU_B
c = SDIRK4_TABLEAU_C
order_values = np.asarray((
    b.sum(),
    b @ c,
    b @ (c**2),
    b @ A @ c,
    b @ (c**3),
    b @ (c * (A @ c)),
    b @ A @ (c**2),
    b @ A @ A @ c,
))
order_targets = np.asarray((1.0, 0.5, 1/3, 1/6, 0.25, 0.125, 1/12, 1/24))
symplecticity_matrix = b[:, None] * A + b[None, :] * A.T - b[:, None] * b[None, :]
symmetry_matrix = A + A[::-1, ::-1] - b[::-1][None, :]
symplecticity_defect = float(np.max(np.abs(symplecticity_matrix)))
symmetry_defect = float(max(
    np.max(np.abs(b - b[::-1])),
    np.max(np.abs(c - (1.0 - c[::-1]))),
    np.max(np.abs(symmetry_matrix)),
))

np.testing.assert_allclose(order_values, order_targets, rtol=0.0, atol=3e-16)
assert np.allclose(np.diag(A), 0.25)
assert np.allclose(A[-1], b)
assert symplecticity_defect > 0.1
assert symmetry_defect > 0.1

display(Markdown(
    f"**Tableau audit:** all order-four conditions pass to `{np.max(np.abs(order_values - order_targets)):.3e}`; "
    f"maximum symplecticity-condition defect = `{symplecticity_defect:.3e}` and "
    f"maximum adjoint-symmetry defect = `{symmetry_defect:.3e}`."
))

**Tableau audit:** all order-four conditions pass to `1.110e-16`; maximum symplecticity-condition defect = `1.250e-01` and maximum adjoint-symmetry defect = `1.750e+00`.

## Aligned integrations and audited reference

All five models receive identical physical data, output times, and effective step. The four implicit methods also share the Newton tolerances; classical RK4 performs no nonlinear solve. Three complete integrations of every method are timed in alternating order and the reported runtime is the median. The three trajectories are advanced together in each vectorized integration.

With `config.progress=True`, the output below is live when the notebook is run interactively. It reports both study-level events and one-percent fixed-step progress. The same stream is written incrementally to `calculation.log`, so remote CLI runs can be followed with `tail -f`.

In [4]:
class ExecutionLogTee:
    """Mirror live stderr output to the notebook and a plain-text log file."""

    def __init__(self, notebook_stream, file_stream):
        self.notebook_stream = notebook_stream
        self.file_stream = file_stream

    def write(self, text):
        self.notebook_stream.write(text)
        # Carriage-return progress bars become readable append-only log lines.
        self.file_stream.write(text.replace("\r", "\n"))
        self.file_stream.flush()
        return len(text)

    def flush(self):
        self.notebook_stream.flush()
        self.file_stream.flush()


print(f"Live calculation log: {execution_log_path.relative_to(project_root)}", flush=True)
print(
    "The cell output shows method/repeat events, campaign ETA, and step progress. "
    "From another shell, follow the same stream with "
    f"`tail -f {execution_log_path.relative_to(project_root)}`.",
    flush=True,
)
with execution_log_path.open("w", encoding="utf-8") as log_stream:
    with redirect_stderr(ExecutionLogTee(sys.stderr, log_stream)):
        result = run_five_method_comparison(
            potential,
            initial_configuration,
            config=config,
        )

abba_diagnostics = result.solutions["ABBA4ImplicitSingleProjection"].diagnostics
gauss_diagnostics = result.solutions["GaussLegendre4"].diagnostics
bm4_diagnostics = result.solutions["BM4Implicit"].diagnostics
sdirk_diagnostics = result.solutions["SDIRK4"].diagnostics
rk4_diagnostics = result.solutions["RK4"].diagnostics
assert abba_diagnostics["projection_formulation"] == "reduced_multiplier"
assert abba_diagnostics["projection_placement"] == "around_complete_composition"
assert abba_diagnostics["nonlinear_solves_per_step"] == 1
assert np.asarray(abba_diagnostics["nonlinear_iterations"]).shape == (config.step_count,)
assert np.asarray(abba_diagnostics["substep_nonlinear_iterations"]).shape == (config.step_count, 1)
assert bm4_diagnostics["projection_solver_formulation"] == "bm4_implicit_reduced"
assert gauss_diagnostics["stage_count"] == 2
assert sdirk_diagnostics["stage_count"] == 5
assert sdirk_diagnostics["symmetric"] is False
assert sdirk_diagnostics["symplectic"] is False
assert np.asarray(sdirk_diagnostics["stage_nonlinear_iterations"]).shape == (config.step_count, 5)
assert all(
    result.solutions[name].diagnostics["nonlinear_solver"] == "newton"
    for name in FIVE_METHOD_IMPLICIT_METHODS
)
assert "nonlinear_solver" not in rk4_diagnostics
assert all(
    result.solutions[name].diagnostics["step_count"] == config.step_count
    for name in FIVE_METHOD_COMPARISON_METHODS
)

display(Markdown(
    f"Study completed in **{result.total_study_runtime_seconds:.3f} s** including both references "
    f"and timed runs. The DOP853/Radau space-time RMS reference discrepancy is "
    f"**{result.reference.time_integrated_rms_floor:.3e}**."
))

Live calculation log: notebooks/developements/energy/compare_five_order4_models_with_sdirk_rk4/calculation.log
The cell output shows method/repeat events, campaign ETA, and step progress. From another shell, follow the same stream with `tail -f notebooks/developements/energy/compare_five_order4_models_with_sdirk_rk4/calculation.log`.


[five-method study] Starting DOP853 reference and tighter Radau audit for 3 trajectories on [0, 200].
[five-method study] Completed both adaptive references in 7353.81 s.
[five-method study] Starting timing 1/3: Single-projection implicit ABBA4; 3 trajectories together, 10000 steps.
ABBA4Implicit [==============================] 100.0% (10000/10000, t=200)
[five-method study] Completed timing 1/3: Single-projection implicit ABBA4 in 810.38 s; campaign 1/15 (6.7%), elapsed 810.4 s, ETA 11345.3 s.
[five-method study] Starting timing 1/3: Gauss--Legendre (2 stages, order 4); 3 trajectories together, 10000 steps.
GaussLegendre4 [==============================] 100.0% (10000/10000, t=200)
[five-method study] Completed timing 1/3: Gauss--Legendre (2 stages, order 4) in 107.61 s; campaign 2/15 (13.3%), elapsed 918.0 s, ETA 5966.9 s.
[five-method study] Starting timing 1/3: Single-projection implicit BM4; 3 trajectories together, 10000 steps.
BM4Implicit [==============================] 100.0%

Study completed in **15882.648 s** including both references and timed runs. The DOP853/Radau space-time RMS reference discrepancy is **2.121e+00**.

## Persist the complete calculation

The CSV is deliberately colocated with the calculation and visualization notebooks. Re-running this cell atomically replaces the previous artifact only after the new file has been written successfully.

In [5]:
written_path = write_five_method_comparison_csv(
    result,
    results_path,
    metadata={
        "study_name": "Five fourth-order models over 200 cycles",
        "potential": {
            "source_path": str(data_path.relative_to(project_root)),
            "magnetic_field": magnetic_field,
            "characteristic_length": characteristic_length,
            "mode_selection": mode_selection,
            "interpolation_order": interpolation_order,
        },
        "initial_conditions": {
            "trajectory_count": trajectory_count,
            "seed": initial_condition_seed,
            "domain_margin_fraction": domain_margin_fraction,
            "near_center_offset_fraction": near_center_offset_fraction,
        },
        "sdirk_tableau_audit": {
            "maximum_order_condition_error": float(
                np.max(np.abs(order_values - order_targets))
            ),
            "symplecticity_defect": symplecticity_defect,
            "adjoint_symmetry_defect": symmetry_defect,
        },
    },
    overwrite=True,
)
assert written_path.is_file()
display(Markdown(
    f"Wrote **{written_path.relative_to(project_root)}** "
    f"({written_path.stat().st_size / 1024**2:.2f} MiB)."
))

Wrote **notebooks/developements/energy/compare_five_order4_models_with_sdirk_rk4/results.csv** (20.00 MiB).